# Advanced Indexing

In [2]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
import getpass

In [3]:
OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')

Enter your OPENAI_API_KEY ········


In [7]:
# Setting up chromadb collections

cornwall_granular_collection = Chroma(
    collection_name = "Cornwall_granular",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

cornwall_granular_collection.reset_collection()

In [8]:
cornwall_coarse_collection = Chroma(
    collection_name = 'cornwall_coarse',
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

cornwall_coarse_collection.reset_collection()

In [10]:
# Loading HTML content with AsyncHTMLLoader

from langchain_community.document_loaders import AsyncHtmlLoader

destination_url = "https://en.wikivoyage.org/wiki/Cornwall"
html_loader = AsyncHtmlLoader(destination_url)
docs = html_loader.load()

Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  3.39it/s]


In [11]:
from langchain_text_splitters import HTMLSectionSplitter

In [13]:
headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(headers_to_split_on=headers_to_split_on)

In [14]:
def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content
        temp_chunks = html_section_splitter.split_text(html_string)
        all_chunks.extend(temp_chunks)

    return all_chunks

In [16]:
granular_chunks = split_docs_into_granular_chunks(docs)
cornwall_granular_collection.add_documents(documents= granular_chunks)

['33792eaa-59ef-4df8-a018-6ee65aeecca6',
 '530ae69f-fa97-4421-8ba4-270908097d8b',
 '156c7750-e357-47cb-85d8-d07e375e8dff',
 'af83cd9c-6360-4493-8229-0ef74f944b63',
 '696d8b54-9f05-4631-b62f-efd930279ec4',
 'dedf2ff6-5289-480a-80fc-9b43f2bdbc44',
 '28b68348-da7c-4a42-8032-ceebab3d317a',
 '604bc574-56b4-40fa-b603-59b37f59b957',
 '76eb78ff-43bc-49a8-b71b-775558cdb88a',
 '33264222-8c33-4c03-9bee-a37ab109554e',
 'aaa98860-4e6b-4c62-a5f2-56cc547aaa75',
 '0b3bf06a-9de6-4540-af0d-c01a38842b1e',
 '0c0bdbec-48ad-4fff-aa09-c4bf6cbf6769',
 '790b4075-24ef-448e-9db3-af24b330bac9',
 '650c3448-be7e-45c5-af7c-c3d875163c1a',
 '87ce2be5-23c5-498c-99d8-d0e8cafdef27',
 '21937148-949c-4d33-a4ba-796164131ddd',
 '6044a1dc-201e-46e2-9f92-94b5bd7a4c7b',
 'ea681c48-a692-455d-ad6e-dfee0b70dae6']

In [18]:
# Searching granular chunks

results = cornwall_granular_collection.similarity_search(query="Events of festivals in Cornwall", k=3)

for doc in results:
    print(doc)

page_content='Festivals 
 [ edit ] 
 
 These festivals tend to not be public holidays and not all are celebrated fully across the county. 
   
 AberFest .   A Celtic cultural festival celebrating “All things” Cornish and Breton that takes place biennially (every two years) in Cornwall at Easter. The AberFest Festival alternates with the Breizh – Kernow Festival that is held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate years.       ( updated Jun 2023 ) 
 Golowan , sometimes also  Goluan  or  Gol-Jowan  is the Cornish word for the Midsummer celebrations, most popular in the Penwith area and in particular  Penzance  and  Newlyn . The celebrations are conducted from the 23rd of June (St John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve being the more popular in Cornish fishing communities. The celebrations are centred around the lighting of bonfires and fireworks and the performance of associated rituals. Some towns have a street-parade dur

In [19]:
# Splitting content into coarse chunks using RecursiveCharacterTextSplitter

from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [21]:
html2text_transformer = Html2TextTransformer()
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 3000, chunk_overlap = 300)

In [26]:
def split_docs_into_coarse_chunks(docs):
    text_docs = html2text_transformer.transform_documents(docs)
    coarse_chunks = text_splitter.split_documents(text_docs)

    return coarse_chunks

In [27]:
coarse_chunks = split_docs_into_coarse_chunks(docs)
cornwall_coarse_collection.add_documents(documents=coarse_chunks)

['a439020a-f264-4dd4-b497-59739413725a',
 '6d05a749-d13e-4520-9a3b-2f8a90010efb',
 'e80e3839-9766-4bf8-a024-55fb31d49578',
 'dba2b3b8-779c-4271-88dd-29c2869f9cd2',
 '1074c85a-fcea-49cd-a8db-0248ee6c0ff1',
 '179eac92-c074-437c-85ac-16e243b67ded',
 '6d644e73-06f8-4c5b-aa17-bafec388e23f',
 '90f12ade-9b7b-4092-a2b2-10e4c427a8b4',
 '5d1b4a89-5e34-43f2-a1a7-23b2a9da8d05',
 '38b383e4-ffc7-4e77-9c37-e88a926b3a4a',
 '35924102-5ae1-4644-9fae-dc60d80afb7d',
 '4e439579-2e0f-41b1-afcb-b9492f02df62',
 '421f0572-1c19-4fa2-a061-7a037f770392',
 '8eb177c3-42ca-4940-aed0-0597de2f971c',
 '216f6b79-9b5c-4c8d-9d9d-6605c2e8377b']

In [29]:
# Searching coarse chunks

results = cornwall_coarse_collection.similarity_search(query="Events or festivals in Cornwall", k=3)

for doc in results:
    print(doc)

page_content='### Spirits

[edit]

    _See also:Liquor_

Gin and rum are also produced in Cornwall. A popular brand of Cornish rum is
Dead Man's Fingers which has multiple flavours and is bottled in St. Ives.

## Festivals

[edit]

These festivals tend to not be public holidays and not all are celebrated
fully across the county.

AberFest. A Celtic cultural festival celebrating “All things” Cornish and
Breton that takes place biennially (every two years) in Cornwall at Easter.
The AberFest Festival alternates with the Breizh – Kernow Festival that is
held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate
years.  (updated Jun 2023)

**Golowan** , sometimes also _Goluan_ or _Gol-Jowan_ is the Cornish word for
the Midsummer celebrations, most popular in the Penwith area and in particular
Penzance and Newlyn. The celebrations are conducted from the 23rd of June (St
John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve
being the more popular in Corn

In [31]:
# Ingesting content from ultiple urls

uk_granular_collection = Chroma(
    collection_name="uk_granular", 
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)
uk_granular_collection.reset_collection()

uk_coarse_collection = Chroma(
    collection_name="uk_coarse",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)
uk_coarse_collection.reset_collection()

uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall",
    "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
    "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven",
    "East_Sussex", "Brighton", "Battle", "Hastings_(England)",
    "Rye_(England)", "Seaford", "Ashdown_Forest"
]


In [34]:
wikivoyage_root_url = "https://en.wikivoyage.org/wiki"
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' for d in uk_destinations]

In [35]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    docs = html_loader.load()

    granular_chunks = split_docs_into_granular_chunks(docs)
    uk_granular_collection.add_documents(documents=granular_chunks)

    coarse_chunks = split_docs_into_coarse_chunks(docs)
    uk_coarse_collection.add_documents(documents=coarse_chunks)

Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  7.90it/s]


In [39]:
granular_results = uk_granular_collection.similarity_search(
query="Events or festivals in East Sussex",k=4)

for doc in granular_results:
    print(doc)

page_content='East Sussex' metadata={'Header 1': 'East Sussex'}
page_content='Go next 
 [ edit ] 
 
 Long Man of Wilmington, on the route of the South Downs Way 
 Attractions outside of East Sussex include: 
 
 
 Tunbridge Wells  (on the A26, signposted from most of the country) - Victorian spa town with bars, pubs and drinking fountains for the local water, is popular in summer with locals and Londoners. Has a large shopping district/center and theaters, worth a day visit. 
 Running from  Eastbourne  in the east all the way to  Petersfield  in the West, spanning three counties the  South Downs Way  is a  popular walking path  with numerous books and guides out there. Walking the full length is completely feasible. Depending on your skill, activity, perseverance, and need for sleep, the path can be completed as quickly as 48 hours (most people take up to a week to complete it). There are outstanding views throughout almost all of the path. Various guide books have been published on the

In [40]:
coarse_results = uk_coarse_collection.similarity_search(
query="Events or festivals in East Sussex",k=4)

for doc in coarse_results:
    print(doc)

page_content='The usual chains of hotels are beginning to spring up.

The towns below have accommodation throughout the year:

  * **Eastbourne** This is one of England’s most famous seaside resorts. The elegant seafront is flanked by flowerbeds. Visitor attractions include parks and gardens, a thriving marina and the cliffs at nearby Beachy Head.
  * **Hastings and St Leonard’s** Popular seaside resorts, surrounded by stunning countryside. Hastings also has a picturesque old town.
  * **Lewes** is one of the county’s oldest towns. Attractions include the castle and Anne of Cleves’ house. Around Lewes there are many picturesque villages to visit.
  * **Rye and surrounding areas** With its steep cobbled streets and picture-postcard cottages, Rye is a charming town. Surrounding attractions include Camber Sands and Winchelsea.
  * **Seaford** is a quiet beach resort. A great base for exploring the South Downs and Seven Sisters Country Park.

_Individual town pages will have more informati

## Embedding Strategy

In [44]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [46]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500)

child_chunks_collection = Chroma(
    collection_name="uk_child_chunks",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)
child_chunks_collection.reset_collection()

doc_store = InMemoryStore()

parent_doc_retriever = ParentDocumentRetriever(
    vectorstore=child_chunks_collection,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

In [49]:
# Ingesting content into document and vector stores

for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    html_docs = html_loader.load()
    text_docs = html2text_transformer.transform_documents(html_docs)

    print(f'Ingesting {destination_url}')
    parent_doc_retriever.add_documents(text_docs, ids=None)

Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  4.17it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  7.80it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  8.34it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  8.83it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  8.01it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [50]:
list(doc_store.yield_keys())

['9d6a9cd5-cd20-4985-8243-48348fc1c171',
 '9d056147-bb9d-4755-8427-40609878c840',
 '3b1dce20-5868-4356-84b0-0825ca8a8a9b',
 '2e2af2c4-c85d-4930-ae57-da3384bccb4c',
 'f93950db-b245-4718-91b5-2cc60a14f1ac',
 '6197e8a3-336c-4d95-bae5-1aff672f90a2',
 'c3fba497-ec2f-44f5-9bbb-7bcc8afe7b5e',
 'd8e494fa-abb3-4e7d-86cb-826e83715301',
 'dd9a5cac-ab3b-4491-8877-2c67fb87d5c9',
 'a9a820e0-772d-4380-8135-e5f2bddfefdd',
 '7b7efbf5-521d-4ccd-a263-6e0a2d8b54a2',
 '80dd1ace-9707-41a2-98bc-ebdb79e95ed0',
 'e9d5064d-cf4c-44e1-814e-e77789a39ef9',
 '2b1c5ae5-140d-45e4-88a6-9b043fc62765',
 'd336711b-ea18-403f-865c-0a30ae42a81b',
 'cd65ebc9-6323-4db7-97f1-5a26cb2bdd5f',
 'b06d9326-7ce2-4893-bb5c-4a017c85adc0',
 '795eea74-5128-4188-8e83-de692045bb2f',
 '1fcef35d-88d1-4f55-9bcf-baa8c3c2baf0',
 'c402820f-b0fe-4243-9b6f-1f79a156bec3',
 '70d46826-d6f4-42a3-8f51-8b5730523603',
 '1c8ed6f5-9066-4843-8df9-356ecb3953d5',
 '44d9ae41-1692-41f6-b8a0-e1f737d14954',
 '3566d3aa-8b18-4e69-a6b1-35cab44b219d',
 '1b4313cc-4790-

In [52]:
# Performing search on granular information

retrieved_docs = parent_doc_retriever.invoke("Cornwall Ranger")
print(retrieved_docs)

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/South_Cornwall', 'title': 'South Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content="## Get around\n\n[edit]\n\n### By bus\n\n[edit]\n\nThanks to Transport for Cornwall, all bus tickets are interchangeable across\nthe different companies. The **Cornwall All Day ticket** allows unlimited\ntravel for a calendar day. As of 2023, fares are £5 for adults and £4 for\nunder-19s. Payment is by cash or contactless. The two main bus companies are:\n\n  * **Go Cornwall Bus** covers all parts of Cornwall and connects with Plymouth (in Devon).\n  * **Kernow** (part of First Bus) covers western and central Cornwall.\n\nBuses only serve designated stops when in towns; otherwise, you can flag them\ndown anywhere that's safe for them to stop.\n\n### By train\n\n[edit]\n\n**CrossCountry Trains** and **Great Western Railway** operate regular train\nservices between the main centres of population, the latter company also\ns

In [54]:
# Comparing with Direct demantic search on child chunks

child_docs_only = child_chunks_collection.similarity_search("Cornwall Ranger")
print(child_docs_only)

[Document(id='8bb601df-539d-4b83-8e5f-1dc3c19847e1', metadata={'source': 'https://en.wikivoyage.org/wiki/South_Cornwall', 'title': 'South Cornwall – Travel guide at Wikivoyage', 'language': 'en', 'doc_id': '3566d3aa-8b18-4e69-a6b1-35cab44b219d'}, page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\n## See\n\n[edit]\n\nThe **Eden Project** , near St Austell, a fabulous collection of flora from\nall over the planet housed in two space age transparent domes, and a massive\nzip line.'), Document(id='73f94a0d-2f33-4168-a260-fc750858386c', metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'doc_id': '2e2af2c4-c85d-4930-ae57-da3384bccb4c', 'language': 'en'}, page_content='### Cornish\n\n[edit]'), Document(id='40a0e4e7-e6cc-4b76-bff9-f8ac585565bd', metadata={'language': 'en', 'title': 'Cornwall – Travel

### Embedding child chunks with MultiVectorRetriever

In [56]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

In [58]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500)

child_chunks_collection = Chroma(
    collection_name="uk_child_chunks",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)
child_chunks_collection.reset_collection()

doc_byte_store = InMemoryStore()
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever(
    vectorstore=child_chunks_collection,
    byte_store=doc_byte_store
)



In [61]:
# Ingesting content into document and vector stores

for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    html_docs = html_loader.load()
    text_docs = html2text_transformer.transform_documents(html_docs)

    coarse_chunks = parent_splitter.split_documents(text_docs)
    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_granular_chunks = []
    for i, coarse_chunk in enumerate(coarse_chunks):
        coarse_chunk_id = coarse_chunks_ids[i]
        granular_chunks = child_splitter.split_documents([coarse_chunk])

        for granular_chunk in granular_chunks:
            granular_chunk.metadata[doc_key]=coarse_chunk_id

        all_granular_chunks.extend(granular_chunks)

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(all_granular_chunks)
    multi_vector_retriever.docstore.mset(list(zip(coarse_chunks_ids, coarse_chunks)))

Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  6.71it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  8.20it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  7.79it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  8.71it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  7.96it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [63]:
# Performing a search on granular information

retrieved_docs = multi_vector_retriever.invoke("Cornwall Ranger")
print(retrieved_docs)

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/South_Cornwall', 'title': 'South Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content="## Get around\n\n[edit]\n\n### By bus\n\n[edit]\n\nThanks to Transport for Cornwall, all bus tickets are interchangeable across\nthe different companies. The **Cornwall All Day ticket** allows unlimited\ntravel for a calendar day. As of 2023, fares are £5 for adults and £4 for\nunder-19s. Payment is by cash or contactless. The two main bus companies are:\n\n  * **Go Cornwall Bus** covers all parts of Cornwall and connects with Plymouth (in Devon).\n  * **Kernow** (part of First Bus) covers western and central Cornwall.\n\nBuses only serve designated stops when in towns; otherwise, you can flag them\ndown anywhere that's safe for them to stop.\n\n### By train\n\n[edit]\n\n**CrossCountry Trains** and **Great Western Railway** operate regular train\nservices between the main centres of population, the latter company also\ns

In [64]:
# Comparing with Direct semantic search on child chunks

child_docs_only = child_chunks_collection.similarity_search("Cornwall Ranger")
print(child_docs_only)

[Document(id='db51b1dc-0221-4929-b7bf-15a59b77ff50', metadata={'doc_id': '60c624d8-fccc-4e34-8b16-86885b4b8d25', 'language': 'en', 'source': 'https://en.wikivoyage.org/wiki/South_Cornwall', 'title': 'South Cornwall – Travel guide at Wikivoyage'}, page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\n## See\n\n[edit]\n\nThe **Eden Project** , near St Austell, a fabulous collection of flora from\nall over the planet housed in two space age transparent domes, and a massive\nzip line.'), Document(id='6021a962-d9df-4ecd-9e0b-e6f1b51e6cf5', metadata={'title': 'Cornwall – Travel guide at Wikivoyage', 'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'language': 'en', 'doc_id': 'f2c3c164-0ce8-4229-b366-189f744f26d6'}, page_content='### Cornish\n\n[edit]'), Document(id='3523c04d-f4cc-4860-8688-e9c46ee258c2', metadata={'source': 'https://en.wikivoyage.org/wiki/Cor

### Embeding document summaries

In [66]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid

In [67]:
# SETTING UP THE MULTIVECTORRETRIEVER
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000)

summaries_collection = Chroma(
    collection_name="uk_summaries",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)
summaries_collection.reset_collection()

doc_byte_store = InMemoryByteStore()
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever(
    vectorstore=summaries_collection,
    byte_store=doc_byte_store
)

In [69]:
# SETTING UP THE SUMMARIZATION CHAIN
llm = ChatOpenAI(model="gpt-5-nano", openai_api_key=OPENAI_API_KEY)
summarization_chain = (
    {"document": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{document}")
    | llm
    | StrOutputParser())

In [70]:
# INGESTING COARSE CHUNKS AND SUMMARIES INTO STORES

for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    html_docs = html_loader.load()
    text_docs = html2text_transformer.transform_documents(html_docs)
    
    coarse_chunks = parent_splitter.split_documents(text_docs)
    
    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_summaries = []
    for i, coarse_chunk in enumerate(coarse_chunks):
        coarse_chunk_id = coarse_chunks_ids[i]
        
        summary_text = summarization_chain.invoke(coarse_chunk)
        summary_doc = Document(page_content=summary_text,
        metadata={doc_key: coarse_chunk_id})
        
        all_summaries.append(summary_doc)
    
    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(all_summaries)
    multi_vector_retriever.docstore.mset(list(zip(coarse_chunks_ids, coarse_chunks)))

Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  4.87it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  5.27it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  8.66it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  3.34it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  8.07it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [71]:
# PERFORMING A SEARCH USING THE MULTIVECTORRETRIEVER
retrieved_docs = multi_vector_retriever.invoke("Cornwall travel")
print(retrieved_docs)

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content="Jump to content\n\nMain menu\n\nMain menu\n\nmove to sidebar hide\n\nNavigation\n\n  * Main page\n  * Travel destinations\n  * Star articles\n  * What's nearby?\n  * Travel forum\n  * Arrivals lounge\n  * Random page\n\nGet involved\n\n  * Travellers' pub\n  * Recent changes\n  * Community portal\n  * Maintenance panel\n  * Policies\n  * Help\n  * Interlingual lounge\n\nSearch\n\nSearch\n\nAppearance\n\n  * Donate\n  * Create account\n  * Log in\n\nPersonal tools\n\n  * Donate\n  * Create account\n  * Log in\n\n## Contents\n\nmove to sidebar hide\n\n  * Beginning\n\n  * 1 Regions\n\n  * 2 Towns and cities\n\n  * 3 Other destinations\n\n  * 4 Understand\n\nToggle Understand subsection\n\n    * 4.1 Visitor information\n\n  * 5 Talk\n\nToggle Talk subsection\n\n    * 5.1 English\n\n    * 5.2 Cornish\n\n  * 6 Get in\n\nToggle Get in sub

In [72]:
# COMPARING WITH DIRECT SEMANTIC SEARCH ON SUMMARIES
summary_docs_only = summaries_collection.similarity_search("Cornwall Travel")
print(summary_docs_only[0])

page_content='- This is a Wikivoyage travel guide page for Cornwall, a county in the West Country of England (Britain and Ireland section).
- It outlines the page structure: Regions; Towns and cities; Other destinations; Understand (including Visitor information); Get in (by plane, ferry, train, car, coach); Get around (by bus, train, ferry/boat); See (National Trust properties and gardens); Do; Eat (Savoury and Sweet); Drink (Ale/beer, Cider, Wine, Mead, Spirits); Festivals; Sleep; Stay safe.
- The page is available in multiple languages (German, Spanish, French, Italian, Polish) and includes a note about disambiguation for places named Cornwall.' metadata={'doc_id': 'ee6c18a2-de9d-49ea-a228-560bbfb317f5'}


### Embedding hypothetical questions

In [73]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid
from typing import List
from pydantic import BaseModel, Field

In [74]:
# SETTING UP THE MULTIVECTORRETRIEVER
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000)

hypothetical_questions_collection = Chroma(
    collection_name="uk_hypothetical_questions",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

hypothetical_questions_collection.reset_collection()

doc_byte_store = InMemoryByteStore()
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever(
    vectorstore=hypothetical_questions_collection,
    byte_store=doc_byte_store
)

In [75]:
# SETTING UP THE HYPOTHETICAL QUESTION GENERATION CHAIN
class HypotheticalQuestions(BaseModel):
    """A list of hypotetical questions for given text."""
    
    questions: List[str] = Field(..., description="List of hypothetical questions for given text")

llm_with_structured_output = ChatOpenAI(
    model="gpt-5-nano",
    openai_api_key=OPENAI_API_KEY).with_structured_output(HypotheticalQuestions
)

In [76]:
hypothetical_questions_chain = (
    {"document_text": lambda x: x.page_content}
    | ChatPromptTemplate.from_template(
        "Generate a list of exactly 4 hypothetical questions that the below text could be used to answer:\n\n{document_text}"
    )
    | llm_with_structured_output
    | (lambda x: x.questions)
)

In [77]:
# INGESTING COARSE CHUNKS AND RELATED HYPOTHETICAL QUESTIONS
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    html_docs = html_loader.load()
    text_docs = html2text_transformer.transform_documents(html_docs)
    
    coarse_chunks = parent_splitter.split_documents(text_docs)
    
    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_hypothetical_questions = []
    for i, coarse_chunk in enumerate(coarse_chunks):
        coarse_chunk_id = coarse_chunks_ids[i]
        
        hypothetical_questions = hypothetical_questions_chain.invoke(coarse_chunk)
        hypothetical_questions_docs = [Document(
            page_content=question, metadata={doc_key: coarse_chunk_id})
            for question in hypothetical_questions]
        
        all_hypothetical_questions.extend(hypothetical_questions_docs)
        
    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
    all_hypothetical_questions)
    multi_vector_retriever.docstore.mset(
    list(zip(coarse_chunks_ids, coarse_chunks)))

Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  2.98it/s]
C:\Users\arvenka\repos\building-agents\ch08\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que... safe' for travelers?"]), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
C:\Users\arvenka\repos\building-agents\ch08\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...atural Beauty (AONB)?']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
C:\Users\arvenka\repos\building-agents\ch08\.venv\Lib\site-packages\pydantic

Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  5.39it/s]
C:\Users\arvenka\repos\building-agents\ch08\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...n for North Cornwall?"]), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
C:\Users\arvenka\repos\building-agents\ch08\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que... Festival since 1962?']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
C:\Users\arvenka\repos\building-agents\ch08\.venv\Lib\site-packages\pydantic

Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  8.79it/s]
C:\Users\arvenka\repos\building-agents\ch08\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que... section of the page?"]), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
C:\Users\arvenka\repos\building-agents\ch08\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que... the TV show Poldark?']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
C:\Users\arvenka\repos\building-agents\ch08\.venv\Lib\site-packages\pydantic

...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  6.16it/s]
C:\Users\arvenka\repos\building-agents\ch08\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...equently does it run?']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
C:\Users\arvenka\repos\building-agents\ch08\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...proximate room rates?']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
C:\Users\arvenka\repos\building-agents\ch08\.venv\Lib\site-packages\pydantic

Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  7.93it/s]
C:\Users\arvenka\repos\building-agents\ch08\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...wn Forest been given?']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
C:\Users\arvenka\repos\building-agents\ch08\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...nd approximate costs?']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
C:\Users\arvenka\repos\building-agents\ch08\.venv\Lib\site-packages\pydantic

Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [78]:
# PERFORMING A SEARCH USING THE MULTIVECTORRETRIEVER
retrieved_docs = multi_vector_retriever.invoke("How can you go to Brighton from London?")
print(retrieved_docs[0])

page_content='Jump to content

Main menu

Main menu

move to sidebar hide

Navigation

  * Main page
  * Travel destinations
  * Star articles
  * What's nearby?
  * Travel forum
  * Arrivals lounge
  * Random page

Get involved

  * Travellers' pub
  * Recent changes
  * Community portal
  * Maintenance panel
  * Policies
  * Help
  * Interlingual lounge

Search

Search

Appearance

  * Donate
  * Create account
  * Log in

Personal tools

  * Donate
  * Create account
  * Log in

## Contents

move to sidebar hide

  * Beginning

  * 1 Understand

Toggle Understand subsection

    * 1.1 Local information

  * 2 Get in

Toggle Get in subsection

    * 2.1 By train

    * 2.2 By car

    * 2.3 By bus

    * 2.4 By plane

  * 3 Get around

Toggle Get around subsection

    * 3.1 By bike

    * 3.2 By bus

    * 3.3 By train

    * 3.4 By taxi

  * 4 See

Toggle See subsection

    * 4.1 Alternative

    * 4.2 Art galleries

    * 4.3 Further out

  * 5 Do

Toggle Do subsection

    * 5.1